# Pre-process metadata

Summary: Pre-processing of metadata: adding alpha diversity metrics to metadata, creating beta diversity metrics tables, estimating the babySQUID per infant per age, estimating microbial rhythmicity per age, and estimating microbial temporal volatility per age. Plot of shannon diversity rarefaction curves along sequencing depths and samples retention.

Author: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import matplotlib.ticker as ticker

from functions_script import load_qza_artifact, process_pcoa, save_df_as_tsv, get_taxonomy, get_tax_per_timepoint, plot_taxa_stacked_bar, compute_distance_to_centroid, compute_sleep_score, fit_cosine_and_plot, fit_cosine_and_plot_scaled

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
seq_data_path = f"{path}/data-sensitive/processed_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# seq_data_path = "../data/processed_data"
# figures_path = "../figures"

In [ ]:
# colors
colors = [plt.cm.Spectral(i/float(6)) for i in range(7)]
timepoints_colors = [colors[0], colors[5], colors[6]]

## Load

In [ ]:
# load samples metadata
md_samples = pd.read_excel(f"{meta_data_path}/raw/metadata.xlsx",
                           index_col=0, 
                           sheet_name='metadata_per_sample')
md_samples = md_samples.sort_values(by=["timepoint", "infant_id"])

print(md_samples.shape)
md_samples.head()

In [ ]:
# load metadata per infant per age/timepoint
md_ages = pd.read_excel(f"{meta_data_path}/raw/metadata.xlsx", 
                         index_col=0, sheet_name='metadata_per_age')
md_ages = md_ages.sort_values(by=["timepoint", "infant_id"])

print(md_ages.shape)
md_ages.head()

In [ ]:
# load alpha diversity metrics
observed_features = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/alpha-diversities/observed_features.qza")
shannon = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/alpha-diversities/shannon.qza")
pielou_e = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/alpha-diversities/pielou_e.qza")
faith_pd = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/alpha-diversities/faith_pd.qza")

In [ ]:
# load beta diversity PCoA coordinates - for volatility
pcs_braycurtis = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/braycurtis.qza")[['Axis 1', 'Axis 2', 'Axis 3']]
pcs_jaccard = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/jaccard.qza")[['Axis 1', 'Axis 2', 'Axis 3']]
pcs_uuf = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/unweighted_unifrac.qza")[['Axis 1', 'Axis 2', 'Axis 3']]
pcs_wuf = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/weighted_unifrac.qza")[['Axis 1', 'Axis 2', 'Axis 3']]

In [ ]:
# load beta diversity PCoA ordinations - for pcoas
pcs_braycurtis_ord = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/braycurtis.qza", case='ordination')
pcs_jaccard_ord = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/jaccard.qza", case='ordination')
pcs_uuf_ord = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/unweighted_unifrac.qza", case='ordination')
pcs_wuf_ord = load_qza_artifact(f"{seq_data_path}/6-diversity-3035/pcoas-beta-div-metrics/weighted_unifrac.qza", case='ordination')

In [ ]:
# load taxonomy-level feature tables
phylum_ft = load_qza_artifact(f"{seq_data_path}/9-comp-3035/phylum-feature-table.qza")
genus_ft = load_qza_artifact(f"{seq_data_path}/9-comp-3035/genus-feature-table.qza")

In [ ]:
# load relative frequency genus feature table
genus_ft_rf = load_qza_artifact(f"{seq_data_path}/8-rel-freq-3035/genus-rel-freq-feature-table.qza")

## Pre-process samples metadata

In [ ]:
# save index as column name
md_samples['sample_id'] = md_samples.index

### Melatonin in stool sample (pg/g)

In [ ]:
# check for extreme melatonin values
extreme = md_samples[md_samples["Melatonin in Stool (pg/g)"] >= 1500]["Melatonin in Stool (pg/g)"]

if not extreme.empty:
    # compute z-score for each extreme melatonin value
    mean = md_samples["Melatonin in Stool (pg/g)"].mean()
    std = md_samples["Melatonin in Stool (pg/g)"].std()

    for val in extreme.values:
        z = (val - mean) / std
        print(f"The value {val} is {z:.2f} standard deviations above the mean.")
else:
    print("No extreme melatonin values found.")

In [ ]:
# set extreme melatonin values to NaN
md_samples["Melatonin in Stool (pg/g)"] = np.where(md_samples["Melatonin in Stool (pg/g)"] >= 1500, np.nan, md_samples["Melatonin in Stool (pg/g)"])

### Alpha diversity per sample

In [ ]:
# merge all metadata and alpha diversity metrics (per sample)
merged_df = md_samples.merge(observed_features, left_index=True, right_index=True)
merged_df = merged_df.merge(shannon, left_index=True, right_index=True)
merged_df = merged_df.merge(pielou_e, left_index=True, right_index=True)
md_samples = merged_df.merge(faith_pd, left_index=True, right_index=True)

## Create beta diversity metrics tables

In [ ]:
# process PCoA dataframes to include metadata
pcs_braycurtis_with_md = process_pcoa(pcs_braycurtis_ord, md_samples)
pcs_jaccard_with_md = process_pcoa(pcs_jaccard_ord, md_samples)
pcs_uuf_with_md = process_pcoa(pcs_uuf_ord, md_samples)
pcs_wuf_with_md = process_pcoa(pcs_wuf_ord, md_samples)

In [ ]:
# save PCoA dataframes with metadata as tsv
save_df_as_tsv(pcs_braycurtis_with_md, f"{meta_data_path}/pcs_braycurtis.tsv")
save_df_as_tsv(pcs_jaccard_with_md, f"{meta_data_path}/pcs_jaccard.tsv")
save_df_as_tsv(pcs_uuf_with_md, f"{meta_data_path}/pcs_uuf.tsv")
save_df_as_tsv(pcs_wuf_with_md, f"{meta_data_path}/pcs_wuf.tsv")

## Pre-process metadata per age/timepoint

### babySQUID and BISQ values

In [ ]:
# define sleep columns
sleep_cols = ['nighttime_sleep_duration_in_h', 'sleep_latency_in_h', 'bedtime_in_h', 'number_of_awakenings']

# check for extreme sleep duration values
md_ages[md_ages[sleep_cols[0]] > 15]

In [ ]:
# check for extreme sleep latency values
md_ages[md_ages[sleep_cols[1]] > 12]

In [ ]:
# set extreme values to NaN
md_ages["nighttime_sleep_duration_in_h"] = np.where(md_ages["nighttime_sleep_duration_in_h"] <= 15, md_ages["nighttime_sleep_duration_in_h"], np.nan)
md_ages["sleep_latency_in_h"] = np.where(md_ages["sleep_latency_in_h"] <= 12, md_ages["sleep_latency_in_h"], np.nan)

In [ ]:
# check cases of missing values in sleep columns
for i in list(range(1, 5)):
    print(md_ages[md_ages[sleep_cols].isna().sum(axis=1) == i].shape[0], "cases of ", i, " missing values")

In [ ]:
# Get the median for each sleep column
sleep_cols_medians = md_ages[sleep_cols].median()

# Identify rows with <= 2 missing values in sleep_cols
mask_fillable = md_ages[sleep_cols].isna().sum(axis=1) <= 2

# Fill ONLY those rows with column medians
md_ages.loc[mask_fillable, sleep_cols] = (md_ages.loc[mask_fillable, sleep_cols].fillna(sleep_cols_medians))

In [ ]:
# compute babySQUID score
md_ages.loc[:, 'babySQUID'] = compute_sleep_score(md_ages)

### Microbial rhythmicity per age/timepoint

#### Alpha diversity rhythmicity

In [ ]:
# timepoints
timepoint_values = md_samples["timepoint"].unique()

In [ ]:
# create cosine fit: observed features and shannon entropy
fig, axs = plt.subplots(2, 2, figsize=(16, 16))

fit_scores_observed, summary_observed = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="observed_features",
    ax=axs[0, 0], 
    show_legend=False)

median_amplitudes_observed = []
median_constants_observed = []
for i, timepoint in enumerate(timepoint_values):
    median_amplitudes_observed.append({timepoint: fit_scores_observed.loc[(fit_scores_observed["R²"] > 0.5) & (fit_scores_observed["timepoint"] == timepoint), "amplitude"].median()})
    median_constants_observed.append({timepoint: fit_scores_observed.loc[(fit_scores_observed["R²"] > 0.5) & (fit_scores_observed["timepoint"] == timepoint), "constant"].median()})

fit_scores_shannon, summary_shannon = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="shannon_entropy",
    ax=axs[0, 1], 
    show_legend=True)

median_amplitudes_shannon = []
median_constants_shannon = []
for i, timepoint in enumerate(timepoint_values):
    median_amplitudes_shannon.append({timepoint: fit_scores_shannon.loc[(fit_scores_shannon["R²"] > 0.5) & (fit_scores_shannon["timepoint"] == timepoint), "amplitude"].median()})
    median_constants_shannon.append({timepoint: fit_scores_shannon.loc[(fit_scores_shannon["R²"] > 0.5) & (fit_scores_shannon["timepoint"] == timepoint), "constant"].median()})

fit_cosine_and_plot_scaled(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors,
    feature_column="observed_features",
    ax=axs[1, 0], 
    fit_scores=fit_scores_observed,
    show_legend=True,
    median_amplitudes=median_amplitudes_observed, 
    median_constants=median_constants_observed)

fit_cosine_and_plot_scaled(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="shannon_entropy",
    ax=axs[1, 1], 
    fit_scores=fit_scores_shannon,
    show_legend=True,
    median_amplitudes=median_amplitudes_shannon, 
    median_constants=median_constants_shannon)

plt.savefig(f"{figures_path}/rhythmicity_waves_1.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# create cosine fit: pielou evenness and faith pd
fig, axs = plt.subplots(2, 2, figsize=(16, 16))

fit_scores_pielou_evenness, summary_pielou_evenness = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="pielou_evenness",
    ax=axs[0, 0], 
    show_legend=False)

median_amplitudes_pielou_evenness = []
median_constants_pielou_evenness = []
for i, timepoint in enumerate(timepoint_values):
    median_amplitudes_pielou_evenness.append({timepoint: fit_scores_pielou_evenness.loc[(fit_scores_pielou_evenness["R²"] > 0.5) & (fit_scores_pielou_evenness["timepoint"] == timepoint), "amplitude"].median()})
    median_constants_pielou_evenness.append({timepoint: fit_scores_pielou_evenness.loc[(fit_scores_pielou_evenness["R²"] > 0.5) & (fit_scores_pielou_evenness["timepoint"] == timepoint), "constant"].median()})

fit_scores_faith_pd, summary_faith_pd = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="faith_pd",
    ax=axs[0, 1], 
    show_legend=True)

median_amplitudes_faith_pd = []
median_constants_faith_pd = []
for i, timepoint in enumerate(timepoint_values):
    median_amplitudes_faith_pd.append({timepoint: fit_scores_faith_pd.loc[(fit_scores_faith_pd["R²"] > 0.5) & (fit_scores_faith_pd["timepoint"] == timepoint), "amplitude"].median()})
    median_constants_faith_pd.append({timepoint: fit_scores_faith_pd.loc[(fit_scores_faith_pd["R²"] > 0.5) & (fit_scores_faith_pd["timepoint"] == timepoint), "constant"].median()})

fit_cosine_and_plot_scaled(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="pielou_evenness",
    fit_scores=fit_scores_pielou_evenness,
    ax=axs[1, 0], 
    show_legend=True,
    median_amplitudes=median_amplitudes_pielou_evenness, 
    median_constants=median_constants_pielou_evenness)

fit_cosine_and_plot_scaled(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="faith_pd",
    fit_scores=fit_scores_faith_pd,
    ax=axs[1, 1], 
    show_legend=True,
    median_amplitudes=median_amplitudes_faith_pd, 
    median_constants=median_constants_faith_pd)

plt.savefig(f"{figures_path}/rhythmicity_waves_2.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# rename R² columns
fit_scores_observed.rename(columns={"R²": "R2_abundance"}, inplace=True)
fit_scores_shannon.rename(columns={"R²": "R2_abundance_evenness"}, inplace=True)
fit_scores_pielou_evenness.rename(columns={"R²": "R2_evenness"}, inplace=True)
fit_scores_faith_pd.rename(columns={"R²": "R2_biodiversity"}, inplace=True)

In [ ]:
# merge R² values into md_ages
merged_df = md_ages.merge(fit_scores_observed[['infant_id', 'timepoint', 'R2_abundance']], on=['infant_id', 'timepoint'], how='left')
merged_df = merged_df.merge(fit_scores_shannon[['infant_id', 'timepoint', 'R2_abundance_evenness']], on=['infant_id', 'timepoint'], how='left')
merged_df = merged_df.merge(fit_scores_pielou_evenness[['infant_id', 'timepoint', 'R2_evenness']], on=['infant_id', 'timepoint'], how='left')
md_ages = merged_df.merge(fit_scores_faith_pd[['infant_id', 'timepoint', 'R2_biodiversity']], on=['infant_id', 'timepoint'], how='left')

#### Individual genera rhythmicity

- Phylum and genus composition along age

In [ ]:
# extract phylum/genus level information from agglom. feature tables
df_phylum = phylum_ft.T.groupby(lambda x: get_taxonomy(x, level='p')).sum().T
df_genus = genus_ft.T.groupby(lambda x: get_taxonomy(x, level='g')).sum().T

In [ ]:
# get the phylum/genus relative abundance per timepoint/age
df_phylum_tp = get_tax_per_timepoint(df_phylum, md_samples, "timepoint")
df_genus_tp = get_tax_per_timepoint(df_genus, md_samples, "timepoint")

In [ ]:
# plot phylum-level stacked barplot over age
plot_taxa_stacked_bar(df_phylum_tp, "phylum", figures_path, threshold=0.01, figsize=(6, 4))

In [ ]:
# plot genus-level stacked barplot over age
plot_taxa_stacked_bar(df_genus_tp, "genus", figures_path, figsize=(7, 4))

- Extract most abundant genera

In [ ]:
# find most abundant genera
print(genus_ft_rf.sum().sort_values(ascending=False).head(6).index.tolist())

In [ ]:
# ['d__Bacteria;p__Firmicutes;c__Negativicutes;o__Veillonellales-Selenomonadales;f__Veillonellaceae;g__Veillonella', 
# 'd__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Escherichia-Shigella', 
# 'd__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Bifidobacterium', 
# 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides',
# 'd__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;__', #ATTENTION: will go into Unassigned
# 'd__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Clostridiaceae;g__Clostridium_sensu_stricto_1']

In [ ]:
# extract genus level information
genus_ft_rf = genus_ft_rf.T.groupby(lambda x: get_taxonomy(x, level='g')).sum().T

In [ ]:
# extract 5 most abundant genera
most_abundant_genera = genus_ft_rf.sum().sort_values(ascending=False).head(6).index.tolist()
most_abundant_genera.remove('Unassigned genus')
print(most_abundant_genera)

In [ ]:
# add abundances of most abundant genera to md_samples
md_samples = md_samples.merge(genus_ft_rf[most_abundant_genera], left_index=True, right_index=True)

- Rhythmicity in (rel. abund. of) most abundant genera

In [ ]:
# create cosine fit: g__Veillonella and g__Bifidobacterium
fit_scores_veillo, summary_veillo = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="g__Veillonella",
    ax=axs[0, 0], 
    show_legend=True)

fit_scores_bifido, summary_bifido = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="g__Bifidobacterium",
    ax=axs[0, 1], 
    show_legend=True)

In [ ]:
# create cosine fit: g__Escherichia-Shigella, g__Bacteroides, and g__Clostridium_sensu_stricto_1
fit_scores_esche, summary_esche = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="g__Escherichia-Shigella",
    ax=axs[0, 0], 
    show_legend=False)

fit_scores_bacteroides, summary_bacteroides = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="g__Bacteroides",
    ax=axs[0, 0], 
    show_legend=False)

fit_scores_clostridium, summary_clostridium = fit_cosine_and_plot(
    data=md_samples, 
    timepoint_values=timepoint_values, 
    timepoints_colors=timepoints_colors, 
    feature_column="g__Clostridium_sensu_stricto_1",
    ax=axs[0, 0], 
    show_legend=False)

In [ ]:
# rename R² columns
fit_scores_veillo.rename(columns={"R²": "R2_veillo"}, inplace=True)
fit_scores_bifido.rename(columns={"R²": "R2_bifido"}, inplace=True)
fit_scores_esche.rename(columns={"R²": "R2_esche"}, inplace=True)
fit_scores_bacteroides.rename(columns={"R²": "R2_bacteroides"}, inplace=True)
fit_scores_clostridium.rename(columns={"R²": "R2_clostridium"}, inplace=True)

In [ ]:
# merge R² values into md_ages
merged_df = md_ages.merge(fit_scores_veillo[['infant_id', 'timepoint', 'R2_veillo']], on=['infant_id', 'timepoint'], how='left')
merged_df = merged_df.merge(fit_scores_bifido[['infant_id', 'timepoint', 'R2_bifido']], on=['infant_id', 'timepoint'], how='left')
merged_df = merged_df.merge(fit_scores_esche[['infant_id', 'timepoint', 'R2_esche']], on=['infant_id', 'timepoint'], how='left')
merged_df = merged_df.merge(fit_scores_bacteroides[['infant_id', 'timepoint', 'R2_bacteroides']], on=['infant_id', 'timepoint'], how='left')
md_ages = merged_df.merge(fit_scores_clostridium[['infant_id', 'timepoint', 'R2_clostridium']], on=['infant_id', 'timepoint'], how='left')

### Microbial temporal volatility per age/timepoint

In [ ]:
# process PCoA dataframes to add infant_id and timepoint
pcs = [pcs_braycurtis, pcs_jaccard, pcs_uuf, pcs_wuf]

for i in range(len(pcs)):
    pcs[i] = pd.merge(pcs[i], md_samples[['infant_id', 'timepoint']], left_index=True, right_index=True, how='left')
    pcs[i]['Sample ID'] = pcs[i].index

In [ ]:
# compute centroids for each infant_id and timepoint
centroids = pcs.copy()

for i in range(len(centroids)):
    centroids[i] = centroids[i].groupby(['infant_id', 'timepoint'])[['Axis 1', 'Axis 2', 'Axis 3']].mean().reset_index()
    centroids[i].rename(columns={'Axis 1': 'Centroid x', 'Axis 2': 'Centroid y', 'Axis 3': 'Centroid z'}, inplace=True)

In [ ]:
# compute distances to centroids and merge into md_samples
new_cols=['distance_to_bc_centroid', 'distance_to_jaccard_centroid', 'distance_to_uuf_centroid', 'distance_to_wuf_centroid']

for i in range(len(new_cols)):
    df = pd.merge(pcs[i], centroids[i], on=['infant_id', 'timepoint'], how='left')
    df.index = df['Sample ID']
    new_col = compute_distance_to_centroid(df, new_cols[i])
    md_samples = pd.merge(md_samples, new_col, left_index=True, right_index=True, how='left')

In [ ]:
# compute volatility scores (median distance to centroid per infant per timepoint)
volatility_scores = pd.DataFrame(md_samples.groupby(['infant_id', 'timepoint'])[new_cols].median())

In [ ]:
# rename volatility score columns
vol_names = ['braycurtis', 'jaccard', 'unweighted_unifrac', 'weighted_unifrac']
for i in range(len(new_cols)):
    volatility_scores.rename(columns={new_cols[i]: f"volatility_{vol_names[i]}"}, inplace=True)
vol_names = volatility_scores.columns.tolist()

# convert index to columns
volatility_scores['infant_id']= volatility_scores.index.get_level_values(0)
volatility_scores['timepoint']= volatility_scores.index.get_level_values(1)
volatility_scores.reset_index(drop=True, inplace=True)

In [ ]:
# merge volatility scores to metadata
md_ages = pd.merge(md_ages, volatility_scores, on=['infant_id', 'timepoint'], how='left')

In [ ]:
# compute number of samples per infant per timepoint
samp_per_subj = (md_samples.groupby(['infant_id', 'timepoint'])
                 .size()
                 .reset_index(name='samples_per_infant_timepoint'))

# merge into original dataframe
md_ages = md_ages.merge(samp_per_subj,
                              on=['infant_id', 'timepoint'],
                              how='left')

In [ ]:
# drop volatility scores when less than 2 samples per infant per timepoint
md_ages.loc[md_ages["samples_per_infant_timepoint"] < 2, vol_names] = np.nan

### Alpha diversity per age/timepoint

In [ ]:
# define alpha diversity columns
alpha_div_cols=['observed_features', 'shannon_entropy', 'pielou_evenness', 'faith_pd']

In [ ]:
# compute median alpha diversity metrics per infant per timepoint
median_alpha_div = md_samples.groupby(['infant_id', 'timepoint'])[alpha_div_cols].median()

In [ ]:
# merge median alpha diversity metrics into md_ages
median_alpha_div['infant_id']= median_alpha_div.index.get_level_values(0)
median_alpha_div['timepoint']= median_alpha_div.index.get_level_values(1)
median_alpha_div.reset_index(drop=True, inplace=True)
md_ages = pd.merge(md_ages, median_alpha_div, on=['infant_id', 'timepoint'], how='left')

## Save processed samples and age metadata tables

### Save processed samples table

In [ ]:
# reassign index
md_samples.index = md_samples['sample_id']

# inspect metadata dataframe
print(md_samples.shape)
md_samples.head()

In [ ]:
# save processed table as tsv
save_df_as_tsv(md_samples, f"{meta_data_path}/md_samples.tsv")

### Save processed age table

In [ ]:
# inspect metadata dataframe
print(md_ages.shape)
md_ages.head()

In [ ]:
# save processed table as tsv
md_ages.to_csv(f"{meta_data_path}/md_ages.tsv", sep='\t', index=True, index_label="id")

## Coverage: rarefaction curves

In [ ]:
# load shannon diversity values computed at each rarefaction depth (20 steps)
shannon_div = pd.read_csv(f"{meta_data_path}/raw/shannon.csv", index_col=0)

In [ ]:
# identify all depth columns and extract the numeric depth value
iter_cols = [c for c in shannon_div.columns if 'depth-' in c and '_iter-' in c]

# extract unique depths
depths = sorted(list(set([int(re.search(r'depth-(\d+)', c).group(1)) for c in iter_cols])))

# compute the mean for each depth (across the 10 iterations)
mean_shannon_div = pd.DataFrame(index=shannon_div.index)
if 'sampleid' in shannon_div.columns:
    mean_shannon_div['sampleid'] = shannon_div['sampleid']

for d in depths:
    # find all 10 columns for this specific depth
    cols_for_depth = [c for c in iter_cols if f'depth-{d}_' in c]
    # calculate row-wise mean and name it as requested
    mean_shannon_div[f'depth-{d}-mean'] = shannon_div[cols_for_depth].mean(axis=1)

In [ ]:
# prepare columns for plotting (sorted by depth)
plot_cols = [f'depth-{d}-mean' for d in depths]

In [ ]:
# plot rarefaction curves with sample retention
fig, ax1 = plt.subplots(figsize=(12, 7))

# plot individual rarefaction curves for each sample
for idx, row in mean_shannon_div.iterrows():
    # only plot if there is at least one non-NaN value
    values = row[plot_cols].values.astype(float)
    if not np.all(np.isnan(values)):
        ax1.plot(depths, values, color='gray', alpha=0.4, linewidth=0.5)

# primary Axis Labels
ax1.set_xlabel('Sequencing Depth', fontsize=12)
ax1.set_ylabel('Mean Shannon Diversity', color='black', fontsize=12)
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True, linestyle='--', alpha=0.5)

# calculate sample retention (how many samples have non-null means at each depth)
samples_retained = mean_shannon_div[plot_cols].notnull().sum(axis=0).values

# create secondary Y-axis for sample retention
ax2 = ax1.twinx()
ax2.step(depths, samples_retained, where='post', color='tab:red', linewidth=2, label='Samples Retained')
ax2.set_ylabel('Number of Samples Retained', color='tab:red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='tab:red')

# light red horizontal line at 163 samples retained
ax2.axhline(y=163, color='lightcoral', linestyle='--', linewidth=1.5, label='n=163 threshold')

# vertical line at depth 3035 (Black)
ax1.axvline(x=3035, color='black', linestyle='-', linewidth=2)

# text box for chosen depth
ax1.text(4000, ax1.get_ylim()[0] + 0.2, 'Chosen depth: 3035', 
         bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'),
         fontsize=10, fontweight='bold')

# text box for chosen # samples
ax1.text(22000, ax1.get_ylim()[1] - 0.2, 'n=163 samples retained', 
         bbox=dict(facecolor='white', edgecolor='coral', boxstyle='round,pad=0.5'),
         fontsize=10, fontweight='bold', color='lightcoral')

# set Y-limit for retention to show the full scale
ax2.set_ylim(0, len(df) + (len(df)*0.1)) 

# set X-axis steps of 2000
ax1.xaxis.set_major_locator(ticker.MultipleLocator(2000))
                            
# set Y-axis steps of 0.5 for Shannon
ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.5))

# set Y-axis steps of 10 for Sample Retention
ax2.yaxis.set_major_locator(ticker.MultipleLocator(10))

fig.tight_layout()

# Save the plot
plt.savefig(f"{figures_path}/rarefaction_mean_plot.pdf", dpi=300, bbox_inches='tight')